In [3]:
# Librerías

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [7]:
# Cargar 

df = pd.read_csv ("kc_house_data.csv")

print("Dimensiones del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores faltantes:")
print(df.isnull().sum())
print("\nEstadísticos descriptivos de price:")
print(df['price'].describe())

Dimensiones del dataset: (21613, 21)

Columnas:
['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15']

Tipos de datos:
id                 int64
date              object
price            float64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object

Valores faltantes:
id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot        

In [23]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [9]:
# Preparación de variables

# Se elimina sqft_living porque es exactamente sqft_above + sqft_basement
# (colinealidad perfecta)
predictors = [
    'bedrooms', 'bathrooms', 'sqft_lot', 'floors', 'waterfront', 
    'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 
    'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15'
]

X = df[predictors].copy()
y = df['price']

print("Variables predictoras seleccionadas:")
print(X.columns.tolist())
print(f"\nNúmero de observaciones: {len(y)}")

Variables predictoras seleccionadas:
['bedrooms', 'bathrooms', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15']

Número de observaciones: 21613


In [11]:
# Enfoque matricial con statsmodels

X_const = sm.add_constant(X)
modelo_completo = sm.OLS(y, X_const).fit()

print(modelo_completo.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.654
Model:                            OLS   Adj. R-squared:                  0.654
Method:                 Least Squares   F-statistic:                     2913.
Date:                Sat, 29 Aug 2026   Prob (F-statistic):               0.00
Time:                        19:51:09   Log-Likelihood:            -2.9614e+05
No. Observations:               21613   AIC:                         5.923e+05
Df Residuals:                   21598   BIC:                         5.924e+05
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const          6.195e+06   1.38e+05     44.754

In [15]:
# Extracción de elementos clave para las 3 pruebas

n = int(modelo_completo.nobs)
k = int(modelo_completo.df_model)
df_resid = int(modelo_completo.df_resid)
alpha = 0.05
t_critico = stats.t.ppf(1 - alpha/2, df_resid)

print(f"n = {n}")
print(f"k (número de predictores) = {k}")
print(f"grados de libertad residuales = {df_resid}")
print(f"α = {alpha}")
print(f"Valor crítico t (bilateral) = {t_critico:.4f}")
print("\n" + "="*60)

# Tabla resumen de coeficientes
resumen = pd.DataFrame({
    'Coeficiente': modelo_completo.params,
    'Error_Estándar': modelo_completo.bse,
    't': modelo_completo.tvalues,
    'p_valor': modelo_completo.pvalues,
    'IC_inf_95%': modelo_completo.conf_int()[0],
    'IC_sup_95%': modelo_completo.conf_int()[1]
})
print("\nTabla de coeficientes del modelo completo:")
print(resumen.round(4))

n = 21613
k (número de predictores) = 14
grados de libertad residuales = 21598
α = 0.05
Valor crítico t (bilateral) = 1.9601


Tabla de coeficientes del modelo completo:
                Coeficiente  Error_Estándar        t  p_valor    IC_inf_95%  \
const          6.195320e+06     138430.5981  44.7540   0.0000  5.923986e+06   
bedrooms      -3.930665e+04       2026.6016 -19.3954   0.0000 -4.327894e+04   
bathrooms      4.574500e+04       3492.3466  13.0986   0.0000  3.889974e+04   
sqft_lot      -1.600000e-03          0.0513  -0.0312   0.9751 -1.021000e-01   
floors         2.687878e+04       3783.8101   7.1036   0.0000  1.946224e+04   
waterfront     5.790716e+05      18636.4310  31.0720   0.0000  5.425428e+05   
view           4.323536e+04       2273.1664  19.0199   0.0000  3.877979e+04   
condition      1.951036e+04       2496.7737   7.8142   0.0000  1.461650e+04   
grade          1.197218e+05       2249.1494  53.2298   0.0000  1.153133e+05   
sqft_above     1.607849e+02          3.9

In [29]:
# Estadístico t vs Región de Rechazo

print("="*70)
print("ENFOQUE 1: Estadístico t vs Región de Rechazo")
print("="*70)
print(f"Región de rechazo (α={alpha}): |t| > {t_critico:.4f}\n")

variables_eliminar_t = []
for var in modelo_completo.params.index:
    if var == 'const':
        continue
    t_val = modelo_completo.tvalues[var]
    decision = "RECHAZAR H0 (relevante)" if abs(t_val) > t_critico else "NO RECHAZAR H0 (no relevante)"
    print(f"{var:15s} | t = {t_val:8.3f} | |t| = {abs(t_val):8.3f} | {decision}")
    if abs(t_val) <= t_critico:
        variables_eliminar_t.append(var)

print(f"\nVariables a eliminar según este enfoque: {variables_eliminar_t}")

ENFOQUE 1: Estadístico t vs Región de Rechazo
Región de rechazo (α=0.05): |t| > 1.9601

bedrooms        | t =  -19.395 | |t| =   19.395 | RECHAZAR H0 (relevante)
bathrooms       | t =   13.099 | |t| =   13.099 | RECHAZAR H0 (relevante)
sqft_lot        | t =   -0.031 | |t| =    0.031 | NO RECHAZAR H0 (no relevante)
floors          | t =    7.104 | |t| =    7.104 | RECHAZAR H0 (relevante)
waterfront      | t =   31.072 | |t| =   31.072 | RECHAZAR H0 (relevante)
view            | t =   19.020 | |t| =   19.020 | RECHAZAR H0 (relevante)
condition       | t =    7.814 | |t| =    7.814 | RECHAZAR H0 (relevante)
grade           | t =   53.230 | |t| =   53.230 | RECHAZAR H0 (relevante)
sqft_above      | t =   41.215 | |t| =   41.215 | RECHAZAR H0 (relevante)
sqft_basement   | t =   35.784 | |t| =   35.784 | RECHAZAR H0 (relevante)
yr_built        | t =  -50.292 | |t| =   50.292 | RECHAZAR H0 (relevante)
yr_renovated    | t =    2.596 | |t| =    2.596 | RECHAZAR H0 (relevante)
sqft_living15   | 

In [19]:
# ENFOQUE 2 - Valores p (significancia)

print("="*70)
print("ENFOQUE 2: Valores de significancia (valores p)")
print("="*70)
print(f"Criterio: rechazar H0 si p-valor < {alpha}\n")

variables_eliminar_p = []
for var in modelo_completo.pvalues.index:
    if var == 'const':
        continue
    p_val = modelo_completo.pvalues[var]
    decision = "RECHAZAR H0 (relevante)" if p_val < alpha else "NO RECHAZAR H0 (no relevante)"
    print(f"{var:15s} | p-valor = {p_val:.6f} | {decision}")
    if p_val >= alpha:
        variables_eliminar_p.append(var)

print(f"\nVariables a eliminar según este enfoque: {variables_eliminar_p}")

ENFOQUE 2: Valores de significancia (valores p)
Criterio: rechazar H0 si p-valor < 0.05

bedrooms        | p-valor = 0.000000 | RECHAZAR H0 (relevante)
bathrooms       | p-valor = 0.000000 | RECHAZAR H0 (relevante)
sqft_lot        | p-valor = 0.975101 | NO RECHAZAR H0 (no relevante)
floors          | p-valor = 0.000000 | RECHAZAR H0 (relevante)
waterfront      | p-valor = 0.000000 | RECHAZAR H0 (relevante)
view            | p-valor = 0.000000 | RECHAZAR H0 (relevante)
condition       | p-valor = 0.000000 | RECHAZAR H0 (relevante)
grade           | p-valor = 0.000000 | RECHAZAR H0 (relevante)
sqft_above      | p-valor = 0.000000 | RECHAZAR H0 (relevante)
sqft_basement   | p-valor = 0.000000 | RECHAZAR H0 (relevante)
yr_built        | p-valor = 0.000000 | RECHAZAR H0 (relevante)
yr_renovated    | p-valor = 0.009443 | RECHAZAR H0 (relevante)
sqft_living15   | p-valor = 0.000000 | RECHAZAR H0 (relevante)
sqft_lot15      | p-valor = 0.000000 | RECHAZAR H0 (relevante)

Variables a eliminar s

In [21]:
# Intervalos de Confianza

print("="*70)
print("ENFOQUE 3: Intervalo de Confianza al 95%")
print("="*70)
print("Criterio: si el IC contiene el 0 → NO se rechaza H0\n")

variables_eliminar_ic = []
ci = modelo_completo.conf_int()
for var in ci.index:
    if var == 'const':
        continue
    inf, sup = ci.loc[var]
    contiene_cero = (inf <= 0 <= sup)
    decision = "NO RECHAZAR H0 (no relevante)" if contiene_cero else "RECHAZAR H0 (relevante)"
    print(f"{var:15s} | IC = [{inf:10.4f}, {sup:10.4f}] | Contiene 0: {contiene_cero} | {decision}")
    if contiene_cero:
        variables_eliminar_ic.append(var)

print(f"\nVariables a eliminar según este enfoque: {variables_eliminar_ic}")

ENFOQUE 3: Intervalo de Confianza al 95%
Criterio: si el IC contiene el 0 → NO se rechaza H0

bedrooms        | IC = [-43278.9411, -35334.3637] | Contiene 0: False | RECHAZAR H0 (relevante)
bathrooms       | IC = [38899.7446, 52590.2589] | Contiene 0: False | RECHAZAR H0 (relevante)
sqft_lot        | IC = [   -0.1021,     0.0989] | Contiene 0: True | NO RECHAZAR H0 (no relevante)
floors          | IC = [19462.2373, 34295.3317] | Contiene 0: False | RECHAZAR H0 (relevante)
waterfront      | IC = [542542.8349, 615600.3962] | Contiene 0: False | RECHAZAR H0 (relevante)
view            | IC = [38779.7877, 47690.9355] | Contiene 0: False | RECHAZAR H0 (relevante)
condition       | IC = [14616.4987, 24404.2203] | Contiene 0: False | RECHAZAR H0 (relevante)
grade           | IC = [115313.3253, 124130.3232] | Contiene 0: False | RECHAZAR H0 (relevante)
sqft_above      | IC = [  153.1384,   168.4315] | Contiene 0: False | RECHAZAR H0 (relevante)
sqft_basement   | IC = [  157.8720,   176.1690] |

In [25]:
# StepWise (Backward Elimination) basado en p-valores

def stepwise_backward(X, y, alpha=0.05):
    """
    Eliminación hacia atrás (backward stepwise) usando p-valores.
    """
    features = list(X.columns)
    historial = []
    
    while True:
        X_temp = sm.add_constant(X[features])
        modelo = sm.OLS(y, X_temp).fit()
        pvalores = modelo.pvalues.drop('const')
        max_p = pvalores.max()
        
        if max_p > alpha:
            var_eliminar = pvalores.idxmax()
            print(f"Eliminando '{var_eliminar}' (p-valor = {max_p:.4f})")
            features.remove(var_eliminar)
            historial.append(var_eliminar)
        else:
            break
    
    return features, modelo, historial

print("="*70)
print("PROCESO STEPWISE (Backward Elimination)")
print("="*70)

features_finales, modelo_final, eliminadas = stepwise_backward(X, y, alpha=0.05)

print(f"\nVariables eliminadas: {eliminadas}")
print(f"Variables retenidas: {features_finales}")

PROCESO STEPWISE (Backward Elimination)
Eliminando 'sqft_lot' (p-valor = 0.9751)

Variables eliminadas: ['sqft_lot']
Variables retenidas: ['bedrooms', 'bathrooms', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15']


In [27]:
# Modelo final

print("="*70)
print("MODELO FINAL DESPUÉS DEL STEPWISE")
print("="*70)
print(modelo_final.summary())

MODELO FINAL DESPUÉS DEL STEPWISE
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.654
Model:                            OLS   Adj. R-squared:                  0.654
Method:                 Least Squares   F-statistic:                     3137.
Date:                Sat, 29 Aug 2026   Prob (F-statistic):               0.00
Time:                        23:32:28   Log-Likelihood:            -2.9614e+05
No. Observations:               21613   AIC:                         5.923e+05
Df Residuals:                   21599   BIC:                         5.924e+05
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const       

In [31]:
# Resumen comparativo de los 3 enfoques

print("Resumen conparativo de los 3 enfoques")

print("\n1. Estadístico t vs Región de rechazo:")
print(f"   Variables eliminadas: {variables_eliminar_t}")

print("\n2. Valores p (α = 0.05):")
print(f"   Variables eliminadas: {variables_eliminar_p}")

print("\n3. Intervalos de confianza al 95%:")
print(f"   Variables eliminadas: {variables_eliminar_ic}")

print("\nConclusión:")
print("Los tres enfoques coinciden: solo se elimina la variable 'sqft_lot'.")
print("El modelo final es el mismo independientemente del criterio utilizado.")

Resumen conparativo de los 3 enfoques

1. Estadístico t vs Región de rechazo:
   Variables eliminadas: ['sqft_lot']

2. Valores p (α = 0.05):
   Variables eliminadas: ['sqft_lot']

3. Intervalos de confianza al 95%:
   Variables eliminadas: ['sqft_lot']

Conclusión:
Los tres enfoques coinciden: solo se elimina la variable 'sqft_lot'.
El modelo final es el mismo independientemente del criterio utilizado.


In [33]:
# Interpretación 

print("""
INTERPRETACIÓN VERBAL DE LOS RESULTADOS
=======================================

• Modelo completo: R² = 0.654 → explica el 65.4% de la variabilidad del precio.

• Variable no relevante: sqft_lot
  - |t| = 0.031 < 1.96 → no se rechaza H0
  - p-valor = 0.975 > 0.05 → no significativa
  - IC 95% = [-0.102, 0.099] contiene el 0

• Todas las demás variables son significativas por los tres criterios.

• Modelo final (después del StepWise):
  price = 6,195,000 
        - 39,310 × bedrooms 
        + 45,740 × bathrooms 
        + 26,880 × floors 
        + 579,100 × waterfront 
        + 43,230 × view 
        + 19,510 × condition 
        + 119,700 × grade 
        + 160.78 × sqft_above 
        + 167.02 × sqft_basement 
        - 3,570 × yr_built 
        + 10.16 × yr_renovated 
        + 24.88 × sqft_living15 
        - 0.552 × sqft_lot15

Interpretación de signos:
- Más metros cuadrados, mejor grado de construcción, vistas y frente al agua → sube el precio.
- Más habitaciones (controlando el tamaño) y casas más antiguas → baja el precio.
""")


INTERPRETACIÓN VERBAL DE LOS RESULTADOS

• Modelo completo: R² = 0.654 → explica el 65.4% de la variabilidad del precio.

• Variable no relevante: sqft_lot
  - |t| = 0.031 < 1.96 → no se rechaza H0
  - p-valor = 0.975 > 0.05 → no significativa
  - IC 95% = [-0.102, 0.099] contiene el 0

• Todas las demás variables son significativas por los tres criterios.

• Modelo final (después del StepWise):
  price = 6,195,000 
        - 39,310 × bedrooms 
        + 45,740 × bathrooms 
        + 26,880 × floors 
        + 579,100 × waterfront 
        + 43,230 × view 
        + 19,510 × condition 
        + 119,700 × grade 
        + 160.78 × sqft_above 
        + 167.02 × sqft_basement 
        - 3,570 × yr_built 
        + 10.16 × yr_renovated 
        + 24.88 × sqft_living15 
        - 0.552 × sqft_lot15

Interpretación de signos:
- Más metros cuadrados, mejor grado de construcción, vistas y frente al agua → sube el precio.
- Más habitaciones (controlando el tamaño) y casas más antiguas → baja